# Lightning Land–Ocean Split (2016–2024)

Purpose: Classify lightning detections over Northern Australia into land and ocean
categories, to support comparison with overshooting-top (OT) detections in the companion
OT notebook.

Input:
- Lightning detection NetCDF files: /g/data/if69/tp4064/lightning_data/<year>/wztln_*_UTC.nc
  (years 2016–2024)
- BARRA-R2 land-fraction file (sftlf):
  .../BARRA-R2/v1/fx/sftlf/latest/sftlf_AUST-11_ERA5_historical_hres_BOM_BARRA-R2_v1.nc
- Region of interest: Northern Australia (lat: -20 to -10, lon: 120 to 148)

Processing:
1. Load and concatenate all yearly lightning files into a single lazy Dataset.
2. Subset to the region of interest and build a lightning-detection mask (lightning_count > 1).
3. Regrid BARRA-R2 land-fraction data (nearest-neighbour) onto the lightning grid.
4. Classify grid cells as land (sftlf > 50%) or ocean (sftlf <= 50%) — threshold chosen
   after testing >0 and >20, both of which caused landmasses to spuriously connect
   (e.g. Australia merging with Papua New Guinea).
5. Apply land/ocean masks to the lightning mask to produce separate detection dataframes.

Output:
- /g/data/if69/tp4064/Project_1/Lightning/land_lightning.feather
- /g/data/if69/tp4064/Project_1/Lightning/ocean_lightning.feather

In [2]:
# Standard libraries
import os
# Data manipulation
import pandas as pd
import numpy as np
import xarray as xr
import glob
import os
import pyarrow


# Parallel computing
import dask
from dask.distributed import Client

# Initialize Dask client with one thread per worker
client = Client(threads_per_worker=1)
client

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.07/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 34291 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/34291/status,
Dashboard: /proxy/34291/status,Workers: 28
Total threads: 28,Total memory: 251.19 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41997,Workers: 0
Dashboard: /proxy/34291/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:36761,Total threads: 1
Dashboard: /proxy/44747/status,Memory: 8.97 GiB
Nanny: tcp://127.0.0.1:44733,


## Region of Interest: Northern Australia

In [3]:
#Intials input
LAT_MAX = -10 #start for upto -10
LAT_MIN = -20 #stop for upto -20
LON_MIN = 120 #start for upto 120
LON_MAX = 148 #stop for upto 148


## Load Lightning Detection Files (2016–2024)
Collects and lazily loads all yearly lightning NetCDF files into a single Dataset.

In [4]:

# Base folder containing yearly folders
base_path = "/g/data/if69/tp4064/lightning_data/"

# List of folders (10 years)
folders = [str(y) for y in range(2016, 2025)]  # adjust end year if needed

# Step 2: Collect all files from all folders
all_files = []
for folder in folders:
    
    folder_path = os.path.join(base_path, folder)
    print(folder_path)
    files = glob.glob(os.path.join(folder_path, "wztln_*_UTC.nc"))
    all_files.extend(files)

# Step 3: Open all files into one Dataset
dataset = xr.open_mfdataset(
    all_files,
    concat_dim='time',
    combine='nested',
    parallel=True,
    chunks={'time': 24, 'grid_lat': 280, 'grid_lon': 280}  # adjust chunk size as needed
)


/g/data/if69/tp4064/lightning_data/2016
/g/data/if69/tp4064/lightning_data/2017
/g/data/if69/tp4064/lightning_data/2018
/g/data/if69/tp4064/lightning_data/2019
/g/data/if69/tp4064/lightning_data/2020
/g/data/if69/tp4064/lightning_data/2021
/g/data/if69/tp4064/lightning_data/2022
/g/data/if69/tp4064/lightning_data/2023
/g/data/if69/tp4064/lightning_data/2024


## Land/Ocean Masking
Subsets lightning data to the region of interest, builds a lightning-detection mask,
and splits it into land vs. ocean using BARRA-R2 land-fraction data.

In [ ]:
# ── Lightning Mask ─────────────────────────────────────────────────────────────
lightning_data_select = dataset.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))
lightning_mask        = (lightning_data_select['lightning_count'] > 1).compute()

# ── Land Fraction ──────────────────────────────────────────────────────────────
FRAC_PATH = ("/g/data/ob53/BARRA2/output/reanalysis/AUST-11/BOM/ERA5/historical/hres/"
             "BARRA-R2/v1/fx/sftlf/latest/sftlf_AUST-11_ERA5_historical_hres_BOM_BARRA-R2_v1.nc")

fraction = (xr.open_dataset(FRAC_PATH)
              .drop_vars("crs"))

# Round consistently to 3dp everywhere
for coord in ['lat', 'lon']:
    lightning_mask[coord] = lightning_mask[coord].round(3)
    fraction[coord]       = fraction[coord].round(3)

fraction_select = (fraction
                   .sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))
                   .interp(lat=lightning_mask['lat'], lon=lightning_mask['lon'], method='nearest'))

l_frac = xr.where(fraction_select['sftlf'] >  50.0, 1, 0)
o_frac = xr.where(fraction_select['sftlf'] <= 50.0, 1, 0)

# ── Apply masks ────────────────────────────────────────────────────────────────
def make_lightning_df(frac_mask, lightning_mask):
    masked = frac_mask.where(lightning_mask).rename('lightning_count')
    df     = masked.to_dataframe().reset_index()
    df     = df.loc[df['lightning_count'] > 0].sort_values('time').reset_index(drop=True)
    return df

lightning_df_land  = make_lightning_df(l_frac, lightning_mask)
lightning_df_ocean = make_lightning_df(o_frac, lightning_mask)

## Save Land/Ocean Lightning DataFrames

In [ ]:
lightning_df_land.to_feather("/g/data/if69/tp4064/Project_1/Lightning/land_lightning.feather")
lightning_df_ocean.to_feather("/g/data/if69/tp4064/Project_1/Lightning/ocean_lightning.feather")